# Test sintetico di pycombat

Questo notebook non usa i dati originali. Genera un dataset sintetico con:

- un effetto clinico vero da preservare (`STABILE` / `INSTABILE`);
- un effetto batch artificiale da rimuovere (`Scanner_A`, `Scanner_B`, `Scanner_C`);
- feature numeriche simili a una matrice radiomica.

Un risultato sensato dovrebbe ridurre le differenze tra batch dopo ComBat, mantenendo il segnale clinico simulato.

In [ ]:
from pathlib import Path
import os
import sys

matplotlib_cache = Path("/tmp/matplotlib")
matplotlib_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_cache))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import bartlett, f_oneway, ttest_ind
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Usa la copia locale del pacchetto pycombat inclusa nel repository.
project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pycombat-radiomics").exists()
)
pycombat_path = project_root / "pycombat-radiomics"
if str(pycombat_path) not in sys.path:
    sys.path.insert(0, str(pycombat_path))

from pycombat import Combat

plt.style.use("seaborn-v0_8-whitegrid")
rng = np.random.default_rng(42)

print(f"Project root: {project_root}")
print(f"pycombat path: {pycombat_path}")

## 1. Generazione dei dati sintetici

Le prime 8 feature avranno un effetto clinico reale. Tutte le feature ricevono invece shift e scala diversi per batch, come se fossero misurate con scanner/protocolli diversi.

In [ ]:
batch_names = np.array(["Scanner_A", "Scanner_B", "Scanner_C"])
n_per_batch = 45
n_features = 40
n_signal_features = 8

feature_names = np.array([f"feature_{idx:02d}" for idx in range(1, n_features + 1)])
batch = np.repeat(batch_names, n_per_batch)

# Bilanciamo STABILE/INSTABILE dentro ogni batch, per non confondere batch e clinica.
clinical_numeric = np.concatenate([
    rng.permutation(np.r_[np.zeros(n_per_batch // 2), np.ones(n_per_batch - n_per_batch // 2)])
    for _ in batch_names
])
clinical_label = np.where(clinical_numeric == 1, "INSTABILE", "STABILE")

baseline = rng.normal(loc=0.0, scale=0.2, size=n_features)
clinical_effect = np.zeros(n_features)
clinical_effect[:n_signal_features] = rng.normal(loc=1.0, scale=0.15, size=n_signal_features)
clinical_effect[:n_signal_features] *= rng.choice([-1, 1], size=n_signal_features)

batch_shift = {
    "Scanner_A": rng.normal(loc=0.0, scale=0.10, size=n_features),
    "Scanner_B": rng.normal(loc=1.2, scale=0.25, size=n_features),
    "Scanner_C": rng.normal(loc=-1.0, scale=0.25, size=n_features),
}
batch_scale = {"Scanner_A": 1.0, "Scanner_B": 1.6, "Scanner_C": 0.65}

Y = np.zeros((batch.size, n_features), dtype=float)
for row_idx, batch_name in enumerate(batch):
    noise = rng.normal(loc=0.0, scale=0.65 * batch_scale[batch_name], size=n_features)
    Y[row_idx] = (
        baseline
        + clinical_effect * clinical_numeric[row_idx]
        + batch_shift[batch_name]
        + noise
    )

metadata = pd.DataFrame({
    "patient_id": [f"SYN_{idx:03d}" for idx in range(1, batch.size + 1)],
    "batch": batch,
    "tipo": clinical_label,
})
features_before = pd.DataFrame(Y, columns=feature_names)
synthetic_df = pd.concat([metadata, features_before], axis=1)

print(synthetic_df.shape)
synthetic_df.head()

## 2. Applicazione di ComBat

Passiamo il batch in `b` e il tipo clinico in `X`, per chiedere a ComBat di rimuovere il batch preservando il segnale `STABILE` / `INSTABILE`.

In [ ]:
X = clinical_numeric.reshape(-1, 1)

combat = Combat()
Y_after = combat.fit_transform(Y=Y, b=batch, X=X)
features_after = pd.DataFrame(Y_after, columns=feature_names)

print("Shape prima:", Y.shape)
print("Shape dopo: ", Y_after.shape)
print("NaN dopo ComBat:", np.isnan(Y_after).sum())
print("Inf dopo ComBat:", np.isinf(Y_after).sum())

## 3. Metriche di controllo

Qui contiamo quante feature risultano ancora associate al batch prima/dopo e quante feature cliniche simulate restano significative.

In [ ]:
def batch_location_pvalues(values: np.ndarray) -> np.ndarray:
    return np.array([
        f_oneway(*(values[batch == batch_name, feature_idx] for batch_name in batch_names))[1]
        for feature_idx in range(values.shape[1])
    ])


def batch_scale_pvalues(values: np.ndarray) -> np.ndarray:
    return np.array([
        bartlett(*(values[batch == batch_name, feature_idx] for batch_name in batch_names))[1]
        for feature_idx in range(values.shape[1])
    ])


def clinical_pvalues(values: np.ndarray) -> np.ndarray:
    return np.array([
        ttest_ind(
            values[clinical_numeric == 0, feature_idx],
            values[clinical_numeric == 1, feature_idx],
            equal_var=False,
        )[1]
        for feature_idx in range(values.shape[1])
    ])


p_batch_before = batch_location_pvalues(Y)
p_batch_after = batch_location_pvalues(Y_after)
p_scale_before = batch_scale_pvalues(Y)
p_scale_after = batch_scale_pvalues(Y_after)
p_clinical_before = clinical_pvalues(Y)
p_clinical_after = clinical_pvalues(Y_after)

signal_mask = np.arange(n_features) < n_signal_features
alpha = 0.05

summary = pd.DataFrame({
    "controllo": [
        "Feature con media diversa tra batch",
        "Feature con varianza diversa tra batch",
        "Feature cliniche simulate ancora significative",
        "Feature non cliniche falsamente significative",
    ],
    "prima": [
        int((p_batch_before < alpha).sum()),
        int((p_scale_before < alpha).sum()),
        int((p_clinical_before[signal_mask] < alpha).sum()),
        int((p_clinical_before[~signal_mask] < alpha).sum()),
    ],
    "dopo": [
        int((p_batch_after < alpha).sum()),
        int((p_scale_after < alpha).sum()),
        int((p_clinical_after[signal_mask] < alpha).sum()),
        int((p_clinical_after[~signal_mask] < alpha).sum()),
    ],
})

summary

## 4. PCA prima/dopo

Se ComBat sta facendo il suo lavoro, dopo l'armonizzazione i punti non dovrebbero piu separarsi soprattutto per scanner.

In [ ]:
def pca_scores(values: np.ndarray) -> np.ndarray:
    scaled = StandardScaler().fit_transform(values)
    return PCA(n_components=2, random_state=0).fit_transform(scaled)


scores_before = pca_scores(Y)
scores_after = pca_scores(Y_after)

colors = {"Scanner_A": "#4c78a8", "Scanner_B": "#f58518", "Scanner_C": "#54a24b"}
markers = {"STABILE": "o", "INSTABILE": "s"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=False, sharey=False)
for ax, scores, title in zip(
    axes,
    [scores_before, scores_after],
    ["Prima di ComBat", "Dopo ComBat"],
):
    for batch_name in batch_names:
        for tipo, marker in markers.items():
            mask = (batch == batch_name) & (clinical_label == tipo)
            ax.scatter(
                scores[mask, 0],
                scores[mask, 1],
                c=colors[batch_name],
                marker=marker,
                s=42,
                alpha=0.78,
                edgecolor="white",
                linewidth=0.5,
                label=f"{batch_name} / {tipo}",
            )
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
fig.tight_layout()
plt.show()

## 5. Distribuzione di una feature per batch

Questo boxplot mostra una feature con segnale clinico simulato. Prima e dopo puoi vedere quanto cambia la separazione per batch.

In [ ]:
feature_to_plot = "feature_01"
feature_idx = int(feature_to_plot.split("_")[1]) - 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, values, title in zip(axes, [Y, Y_after], ["Prima", "Dopo"]):
    box_data = [values[batch == batch_name, feature_idx] for batch_name in batch_names]
    ax.boxplot(box_data, tick_labels=batch_names, patch_artist=True)
    ax.set_title(f"{title}: {feature_to_plot}")
    ax.set_xlabel("Batch")
    ax.set_ylabel("Valore feature")
    ax.tick_params(axis="x", rotation=20)

fig.tight_layout()
plt.show()

## 6. P-value batch e clinica prima/dopo

La linea tratteggiata indica `p = 0.05`. Le prime 8 feature sono quelle con effetto clinico simulato.

In [ ]:
def neglog10(pvalues: np.ndarray) -> np.ndarray:
    return -np.log10(np.clip(pvalues, 1e-300, 1.0))


x = np.arange(1, n_features + 1)
threshold = -np.log10(alpha)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

axes[0].plot(x, neglog10(p_batch_before), label="Prima", color="#d62728", linewidth=1.8)
axes[0].plot(x, neglog10(p_batch_after), label="Dopo", color="#1f77b4", linewidth=1.8)
axes[0].axhline(threshold, color="black", linestyle="--", linewidth=1)
axes[0].axvspan(0.5, n_signal_features + 0.5, color="#dddddd", alpha=0.35)
axes[0].set_title("Associazione feature-batch")
axes[0].set_xlabel("Feature")
axes[0].set_ylabel("-log10(p ANOVA batch)")
axes[0].legend(frameon=False)

axes[1].plot(x, neglog10(p_clinical_before), label="Prima", color="#d62728", linewidth=1.8)
axes[1].plot(x, neglog10(p_clinical_after), label="Dopo", color="#1f77b4", linewidth=1.8)
axes[1].axhline(threshold, color="black", linestyle="--", linewidth=1)
axes[1].axvspan(0.5, n_signal_features + 0.5, color="#dddddd", alpha=0.35)
axes[1].set_title("Associazione feature-clinica")
axes[1].set_xlabel("Feature")
axes[1].set_ylabel("-log10(p t-test tipo)")
axes[1].legend(frameon=False)

fig.tight_layout()
plt.show()

## Come leggere il risultato

- Se ComBat funziona, il numero di feature associate al batch deve scendere molto.
- Le feature cliniche simulate, cioe le prime 8, dovrebbero restare in buona parte significative.
- La PCA dopo ComBat dovrebbe mostrare meno separazione per scanner.
- Se passi il tipo clinico in `X`, stai dicendo a ComBat: rimuovi il batch, ma preserva questa differenza.